# rag system using langchain and ollama : 

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma

C:\Users\mouad\AppData\Local\Temp\ipykernel_8288\395504171.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


* load data : 

In [3]:
loader = TextLoader("../data/documents/rag_notes.txt")

documents = loader.load()

print("Number of documents:", len(documents))

Number of documents: 1


In [ ]:
print(documents[0].page_content)

print("\nMetadata:")
print(documents[0].metadata)

* Text Splitter : 

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [6]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)


--- Chunk 1 ---
Retrieval-Augmented Generation (RAG) is a technique used in Generative AI.

RAG allows an AI system to retrieve relevant information from external documents before generating an answer.

--- Chunk 2 ---
A typical RAG pipeline contains document loading, text splitting, embeddings, vector storage, retrieval, and generation.

--- Chunk 3 ---
The main advantage of RAG is that the language model can use information from a specific knowledge base.


* Embedding Model : 

In [7]:
embedding_model = OllamaEmbeddings(
    
 model="nomic-embed-text"
)
print("Embedding model is ready!")

Embedding model is ready!


* Chroma Vector Database : 

In [8]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../chroma_db"
)

print("Vector database created successfully!")

Vector database created successfully!


* Retriever : 

In [16]:

#~ search the best 2 answer for user question ->

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever is ready!")

Retriever is ready!


In [17]:
# test retriever : 
question = "What is RAG?"

relevant_chunks = retriever.invoke(question)

print("Number of retrieved chunks:", len(relevant_chunks))

Number of retrieved chunks: 3


In [18]:
for i, chunk in enumerate(relevant_chunks):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(chunk.page_content)


--- Retrieved Chunk 1 ---
The main advantage of RAG is that the language model can use information from a specific knowledge base.

--- Retrieved Chunk 2 ---
A typical RAG pipeline contains document loading, text splitting, embeddings, vector storage, retrieval, and generation.

--- Retrieved Chunk 3 ---
Retrieval-Augmented Generation (RAG) is a technique used in Generative AI.

RAG allows an AI system to retrieve relevant information from external documents before generating an answer.


* set llm : 

In [19]:
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0.3
)

print("LLM is ready!")

LLM is ready!


* initial the prompt template : 

In [20]:
from langchain_core.prompts import ChatPromptTemplate

# make the llm answer only base on knowledge base : 

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using only the context provided below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

* Rag func : 

In [21]:
def ask_rag(question):
    
    # 1. Retrieve relevant chunks
    relevant_chunks = retriever.invoke(question)

    # 2. Combine chunks into one context
    context = "\n\n".join(
        chunk.page_content
        for chunk in relevant_chunks
    )

    # 3. Create the prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # 4. Ask Qwen3
    response = llm.invoke(messages)

    return response.content

In [23]:

#* test the rag 
answer = ask_rag("who is mouad ?")

print(answer)

I don't know based on the provided documents.
